In [1]:
!pip install -r ../requirements.txt

In [5]:
import os
import pickle
import time
import json
import numpy as np
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

In [3]:
CHUNKS_PATH = "../data/processed/chunks.pkl"

with open(CHUNKS_PATH, "rb") as f:
    chunks = pickle.load(f)

print(f"✅ Loaded {len(chunks)} chunks")
print(f"\nSample chunk:")
print(f"  Source  : {chunks[100].metadata.get('source', 'Unknown')}")
print(f"  Page    : {chunks[100].metadata.get('page', 'Unknown')}")
print(f"  Length  : {len(chunks[100].page_content)} chars")
print(f"  Content : {chunks[100].page_content[:200]}...")


✅ Loaded 2997 chunks

Sample chunk:
  Source  : ../data/raw/NSCA_1.pdf
  Page    : 1
  Length  : 501 chars
  Content : ing these issues in negligence cases
have ruled that violations of such pro-
fessional standards often constitute
a breach of duty.
If properly adopted and applied, pub-
lished standards of practice
c...


In [10]:
MODEL_NAME = "multi-qa-MiniLM-L6-cos-v1"

print(f"Loading embedding model: {MODEL_NAME}")
print("First run downloads ~80MB — subsequent runs load from cache\n")

start = time.time()

embedding_model = HuggingFaceEmbeddings(
    model_name=MODEL_NAME,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

elapsed = time.time() - start

# Quick test
test_vec = embedding_model.embed_query("How many days should a beginner train per week?")

print(f"✅ Model loaded in {elapsed:.1f}s")
print(f"   Embedding dimensions : {len(test_vec)}")
print(f"   Sample values        : {np.round(test_vec[:6], 4)}")
print(f"   Vector norm          : {np.linalg.norm(test_vec):.4f}")


Loading embedding model: multi-qa-MiniLM-L6-cos-v1
First run downloads ~80MB — subsequent runs load from cache



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7648.42it/s]


✅ Model loaded in 4.8s
   Embedding dimensions : 384
   Sample values        : [-0.0062 -0.002  -0.0092  0.0635 -0.0528  0.0514]
   Vector norm          : 1.0000


In [23]:
INDEX_PATH = "../embeddings/vector_store"
os.makedirs(INDEX_PATH, exist_ok=True)

print(f"Building FAISS index for {len(chunks)} chunks...")
print("Processing in batches — please wait...\n")

start = time.time()

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

elapsed = time.time() - start

print(f"✅ FAISS index built in {elapsed:.1f}s")
print(f"   Total vectors indexed : {vectorstore.index.ntotal}")
print(f"   Embedding dimensions  : {vectorstore.index.d}")
print(f"   Index type            : {type(vectorstore.index).__name__}")


Building FAISS index for 2997 chunks...
Processing in batches — please wait...

✅ FAISS index built in 39.0s
   Total vectors indexed : 2997
   Embedding dimensions  : 384
   Index type            : IndexFlatL2


In [24]:
vectorstore.save_local(INDEX_PATH)

saved_files = os.listdir(INDEX_PATH)
print(f"✅ Vector store saved to '{INDEX_PATH}/'")
print(f"   Files: {saved_files}")
print(f"\n   To reload in any notebook:")
print(f"   vectorstore = FAISS.load_local(")
print(f"       '{INDEX_PATH}',")
print(f"       embedding_model,")
print(f"       allow_dangerous_deserialization=True")
print(f"   )")


✅ Vector store saved to '../embeddings/vector_store/'
   Files: ['index.faiss', 'index.pkl']

   To reload in any notebook:
   vectorstore = FAISS.load_local(
       '../embeddings/vector_store',
       embedding_model,
       allow_dangerous_deserialization=True
   )


In [41]:
def show_results(query, docs, label=""):
    print(f"{'='*65}")
    print(f"  Query  : {query}")
    if label:
        print(f"  Method : {label}")
    print(f"{'='*65}")
    for i, doc in enumerate(docs):
        src  = os.path.basename(doc.metadata.get("source", "Unknown"))
        page = doc.metadata.get("page", "?")
        print(f"\n  Result #{i+1} | {src} — page {page} | {len(doc.page_content)} chars")
        print(f"  {doc.page_content[:200]}...")
    print()

standard_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

q1 = "How many days per week should a beginner train?"
show_results(q1, standard_retriever.invoke(q1), "Standard cosine similarity (k=5)")


  Query  : How many days per week should a beginner train?
  Method : Standard cosine similarity (k=5)

  Result #1 | SSW.pdf — page 2 | 483 chars
  Who Wants to Be a Novice?
3 StartingStrength.com© 2013 The Aasgaard Company
so he’s not really capable of inflicting enough training stress in a sane workout to prevent his recovery 
in a short period...

  Result #2 | Increasing_Anaerobic_Endurance_Using_Strength_Endu.pdf — page 6 | 468 chars
  training cycle that uses a weekly timeframe, also known as weekly training (Lorenz & Morrison, 
2015). This study used continuous running with power endurance and continuous running with 
muscular end...

  Result #3 | NSCA_5.pdf — page 11 | 479 chars
  movements that should be performed at a high velocity.
Although additional research is needed, it is likely that the
performance of different training velocities within a training
program may provide ...

  Result #4 | WHO.pdf — page 21 | 509 chars
  emphasizes functional 
balance and strength 
trai

In [40]:
q2 = "What is progressive overload and why does it matter?"
show_results(q2, standard_retriever.invoke(q2), "Standard cosine similarity (k=5)")


  Query  : What is progressive overload and why does it matter?
  Method : Standard cosine similarity (k=5)

  Result #1 | ProgressiveOverloadinLong-TermExerciseInterventionsTargetingExecutiveFunction.pdf — page 5 | 479 chars
  For Peer Review
PROGRESSIVE OVERLOAD AND EXECUTIVE FUNCTION 5
1 forms of structured physical activity and sport that are designed to promote adaptation and enhance EF. 
2 Progressive overload is typic...

  Result #2 | ProgressiveOverloadinLong-TermExerciseInterventionsTargetingExecutiveFunction.pdf — page 19 | 466 chars
  For Peer Review
PROGRESSIVE OVERLOAD AND EXECUTIVE FUNCTION 19
1 progressive overload as an independent variable represent key gaps in the literature. Additionally, few 
2 studies have examined the ap...

  Result #3 | progressive_overload.pdf — page 4 | 494 chars
  improvements based on progressive overload
is individual specific. In some cases,
individuals are not comfortable with
increasing speed and resistance and doing so
may reduce parti

In [28]:
# Out-of-scope query — expected to show retrieval stress
q3 = "What should I eat before a workout?"
show_results(q3, standard_retriever.invoke(q3), "Standard cosine similarity (k=5) — nutrition query (out-of-scope)")


  Query  : What should I eat before a workout?
  Method : Standard cosine similarity (k=5) — nutrition query (out-of-scope)

  Result #1 | NSCA_5.pdf — page 10 | 504 chars
  proper exercise technique.
There are many ways to arrange the sequence of exercises
in a resistance training session. Most youth will perform total
body workouts several times per week, which involve
...

  Result #2 | WHO.pdf — page 19 | 503 chars
  activity, across the week.
Strong recommendation, moderate certainty evidence
 Vigorous-intensity aerobic 
activities, as well as those that 
strengthen muscle and bone 
should be incorporated at leas...

  Result #3 | WHO.pdf — page 20 | 503 chars
  activity, across the week.
Strong recommendation, moderate certainty evidence
 Vigorous-intensity aerobic 
activities, as well as those that 
strengthen muscle and bone 
should be incorporated at leas...

  Result #4 | SSW.pdf — page 2 | 460 chars
  an empty bar doing sets of 5, and go up in small jumps. When you reach a w

In [31]:
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k"           : 5,
        "fetch_k"     : 20,
        "lambda_mult" : 0.5
    }
)

q1 = "How many days per week should a beginner train?"
show_results(q1, mmr_retriever.invoke(q1), "MMR (k=5, fetch_k=20, lambda=0.5)")


  Query  : How many days per week should a beginner train?
  Method : MMR (k=5, fetch_k=20, lambda=0.5)

  Result #1 | SSW.pdf — page 2 | 483 chars
  Who Wants to Be a Novice?
3 StartingStrength.com© 2013 The Aasgaard Company
so he’s not really capable of inflicting enough training stress in a sane workout to prevent his recovery 
in a short period...

  Result #2 | NSCA_5.pdf — page 11 | 479 chars
  movements that should be performed at a high velocity.
Although additional research is needed, it is likely that the
performance of different training velocities within a training
program may provide ...

  Result #3 | Increasing_Anaerobic_Endurance_Using_Strength_Endu.pdf — page 6 | 468 chars
  training cycle that uses a weekly timeframe, also known as weekly training (Lorenz & Morrison, 
2015). This study used continuous running with power endurance and continuous running with 
muscular end...

  Result #4 | WHO.pdf — page 14 | 223 chars
  emphasizes functional 
balance and strength 
tra

In [32]:
def diversity_report(query, std_ret, mmr_ret):
    std_docs = std_ret.invoke(query)
    mmr_docs = mmr_ret.invoke(query)

    def analyse(docs, label):
        sources = [os.path.basename(d.metadata.get("source","?")) for d in docs]
        pages   = [d.metadata.get("page","?") for d in docs]
        unique  = len(set(sources))
        print(f"  {label}")
        for i, (s, p) in enumerate(zip(sources, pages)):
            print(f"    #{i+1} {s} — p.{p}")
        print(f"    Unique source documents: {unique}/{len(docs)}")
        return unique

    print(f"Query: '{query}'")
    print("="*65)
    std_u = analyse(std_docs, "Standard top-k")
    print()
    mmr_u = analyse(mmr_docs, "MMR")
    print(f"\n  MMR improvement: {mmr_u - std_u:+d} unique sources")
    print()

diversity_report(
    "What are the benefits of resistance training for older adults?",
    standard_retriever,
    mmr_retriever
)

diversity_report(
    "How should a beginner structure their weekly workout?",
    standard_retriever,
    mmr_retriever
)


Query: 'What are the benefits of resistance training for older adults?'
  Standard top-k
    #1 NSCA_3.pdf — p.0
    #2 NSCA_3.pdf — p.11
    #3 NSCA_3.pdf — p.1
    #4 NSCA_3.pdf — p.19
    #5 NSCA_5.pdf — p.13
    Unique source documents: 2/5

  MMR
    #1 NSCA_3.pdf — p.0
    #2 NSCA_3.pdf — p.1
    #3 NSCA_5.pdf — p.0
    #4 NSCA_5.pdf — p.9
    #5 NSCA_3.pdf — p.10
    Unique source documents: 2/5

  MMR improvement: +0 unique sources

Query: 'How should a beginner structure their weekly workout?'
  Standard top-k
    #1 WHO.pdf — p.19
    #2 WHO.pdf — p.20
    #3 WHO.pdf — p.21
    #4 WHO.pdf — p.22
    #5 NSCA_5.pdf — p.10
    Unique source documents: 2/5

  MMR
    #1 WHO.pdf — p.19
    #2 SSW.pdf — p.2
    #3 SSW.pdf — p.2
    #4 NSCA_5.pdf — p.10
    #5 NSCA_3.pdf — p.6
    Unique source documents: 4/5

  MMR improvement: +2 unique sources



In [34]:
def score_report(query, k=5):
    results = vectorstore.similarity_search_with_score(query, k=k)
    print(f"Query: '{query}'")
    print(f"{'='*65}")
    for i, (doc, raw_score) in enumerate(results):
        # With normalised vectors: cosine similarity = 1 - (L2_distance^2 / 2)
        cosine = float(1 - (raw_score / 2))
        src    = os.path.basename(doc.metadata.get("source","?"))
        page   = doc.metadata.get("page","?")
        bar    = "█" * int(cosine * 20)
        print(f"  #{i+1} [{bar:<20}] {cosine:.3f} | {src} p.{page}")
    print()

# In-scope queries — expect high scores
score_report("How many sets should a beginner do per exercise?")
score_report("What is the recommended rest time between sets?")

# Out-of-scope — expect low scores (nutrition not in knowledge base)
score_report("What foods should I avoid to lose weight?")


Query: 'How many sets should a beginner do per exercise?'
  #1 [██████████████      ] 0.701 | NSCA_3.pdf p.6
  #2 [████████████        ] 0.601 | progressive_overload.pdf p.2
  #3 [███████████         ] 0.583 | NSCA_3.pdf p.4
  #4 [███████████         ] 0.581 | NSCA_5.pdf p.11
  #5 [███████████         ] 0.571 | NSCA_5.pdf p.11

Query: 'What is the recommended rest time between sets?'
  #1 [███████████         ] 0.565 | NSCA_4.pdf p.9
  #2 [█████████           ] 0.482 | NSCA_5.pdf p.11
  #3 [█████████           ] 0.456 | NSCA_5.pdf p.11
  #4 [████████            ] 0.427 | SSW.pdf p.2
  #5 [████████            ] 0.423 | NSCA_5.pdf p.11

Query: 'What foods should I avoid to lose weight?'
  #1 [█████               ] 0.261 | NSCA_3.pdf p.16
  #2 [████                ] 0.241 | NSCA_3.pdf p.5
  #3 [████                ] 0.241 | NSCA_4.pdf p.1
  #4 [████                ] 0.236 | ProgressiveOverloadinLong-TermExerciseInterventionsTargetingExecutiveFunction.pdf p.28
  #5 [████                ] 0

In [37]:
config = {
    "model_name"     : MODEL_NAME,
    "index_path"     : INDEX_PATH,
    "search_type"    : "mmr",
    "k"              : 5,
    "fetch_k"        : 20,
    "lambda_mult"    : 0.5,
    "total_chunks"   : len(chunks),
    "total_vectors"  : int(vectorstore.index.ntotal),
    "embedding_dims" : int(vectorstore.index.d)
}

config_path = "../data/processed/retriever_config.json"
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print("✅ Retriever config saved to", config_path)
print(json.dumps(config, indent=2))


✅ Retriever config saved to ../data/processed/retriever_config.json
{
  "model_name": "multi-qa-MiniLM-L6-cos-v1",
  "index_path": "../embeddings/vector_store",
  "search_type": "mmr",
  "k": 5,
  "fetch_k": 20,
  "lambda_mult": 0.5,
  "total_chunks": 2997,
  "total_vectors": 2997,
  "embedding_dims": 384
}
